<a href="https://colab.research.google.com/github/tburleyinfo/vLLM-Hook/blob/codex/MLR-20-eprime-wandb-integration/notebooks/demo_spotlight_e_prime_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Spotlight E-Prime Constraint Retention In Colab [C0_C1]

Experimental iteration copied from the frozen A0-A4 provenance notebook. C0 and C1 intentionally share the same position-level attention intermediate before the canonical Spotlight global aggregation boundary.

- `C0_GLOBAL_DIAGNOSTIC`: record per-query `psi_current(i)` immediately before the existing global aggregation, then continue with canonical global Spotlight behavior.
- `C1_PER_QUERY_CONTROL`: branch from that same `psi_current(i)` intermediate and preserve the query dimension for query-indexed control.

Keep this combined notebook split only if later worker/runtime validation shows the C1 branch forces structural changes that prevent C0 from remaining a canonical global control.


### Installation

Run this setup cell once in a fresh Colab GPU runtime before continuing. It clones the repo and installs the CUDA-compatible notebook dependencies.


In [1]:
# ==============================================================================
# VLLM HOOK SETUP AND DEPENDENCY MANAGER
# ==============================================================================
#
# PURPOSE:
# This cell prepares the environment to run vLLM and its custom plugins.
# It handles complex dependency conflicts common in Colab (e.g., pre-installed
# torch versions) and ensures the plugin code is loaded correctly.
#
# KEY ACTIONS:
# 1. Clones the vLLM-Hook repository if not present.
# 2. Installs a compatible version of vLLM and PyTorch for your GPU.
# 3. Cleans up old binary artifacts to prevent version conflicts.
# 4. Loads the plugin source code.
# 5. TRIGGERS A COLAB RESTART: The cell will restart the runtime to ensure
#    the new libraries are fully loaded into the kernel memory.
#
# EXPECTED BEHAVIOR:
# - The cell will run and install packages.
# - It may print "Restarting Colab runtime...".
# - The cell will stop abruptly, and the runtime will restart.
# - The restart is automatic; the code will resume from the top.
# ==============================================================================

from pathlib import Path
import importlib
import importlib.util
import os
import re
import shutil
import site
import subprocess
import sys
import time

# Configuration
REPO_URL = os.environ.get("VLLM_HOOK_REPO_URL", "https://github.com/tburleyinfo/vLLM-Hook.git")
REPO_BRANCH = os.environ.get("VLLM_HOOK_REPO_BRANCH", "codex/MLR-20-eprime-wandb-integration")
REPO_NAME = "vLLM-Hook"

# Environment Variables (Optional overrides)
COLAB_INSTALL_VLLM = os.environ.get("COLAB_INSTALL_VLLM", "")
VLLM_SPEC = os.environ.get("VLLM_SPEC", "vllm>=0.11,<0.19")
VLLM_TORCH_BACKEND = os.environ.get("VLLM_TORCH_BACKEND", "cu128")

IN_COLAB = "google.colab" in sys.modules
NOTEBOOK_DIR = Path.cwd()

# --------------------------------------------------------------------------
# Helper Functions
# --------------------------------------------------------------------------

def run(cmd, cwd=None, env=None):
    """
    Executes a shell command, printing output in real-time.
    Raises an error if the command fails.
    """
    cmd = [str(part) for part in cmd]
    print(f"> Running: {' '.join(cmd)}", flush=True)

    process = subprocess.Popen(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    tail = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
        tail = tail[-120:]

    returncode = process.wait()
    if returncode:
        tail_text = "\n".join(tail)
        raise RuntimeError(
            f"Command failed with exit code {returncode}: {' '.join(cmd)}\n\n"
            f"Last output lines:\n{tail_text}"
        )

def run_capture(cmd):
    """Executes a command and returns stdout/stderr without printing."""
    return subprocess.run([str(part) for part in cmd], text=True, capture_output=True, check=False)

def norm(name):
    """Normalize package names for comparison."""
    return name.lower().replace("_", "-")

def package_from_req_line(line: str) -> str:
    """Extract package name from a requirement string (e.g., 'torch>=1.0' -> 'torch')."""
    stripped = line.strip()
    # Split on version specifiers
    package = re.split(r"==|>=|<=|~=|!=|<|>|\[", stripped, maxsplit=1)[0]
    return norm(package.strip())

def _repo_remote_matches(repo_root: Path, expected_remote: str) -> bool:
    """Checks if the git repo's origin matches the expected URL."""
    try:
        url = subprocess.run(
            ["git", "-C", str(repo_root), "remote", "get-url", "origin"],
            check=True,
            capture_output=True,
            text=True,
        ).stdout.strip().removesuffix(".git")
    except Exception:
        return False
    return url == expected_remote

def _find_existing_repo_root(start_dir: Path, expected_remote: str):
    """Searches up the directory tree for a matching git repo."""
    for candidate in [start_dir, *start_dir.parents]:
        if (candidate / ".git").exists() and _repo_remote_matches(candidate, expected_remote):
            return candidate
    return None

def assert_cuda_runtime():
    """Ensures a GPU is available before proceeding."""
    try:
        import torch
    except Exception:
        torch = None

    has_cuda = bool(torch is not None and torch.cuda.is_available())
    has_cudart = importlib.util.find_spec("nvidia.cuda_runtime") is not None

    if not has_cuda and not has_cudart:
        raise RuntimeError(
            "No CUDA GPU detected. "
            "In Colab, go to Runtime > Change runtime type and select T4 GPU (or better), "
            "then re-run the entire notebook from the beginning."
        )

# --------------------------------------------------------------------------
# 1. Repository Setup
# --------------------------------------------------------------------------

expected_remote = REPO_URL.removesuffix(".git")
existing_repo_root = _find_existing_repo_root(NOTEBOOK_DIR, expected_remote)

if IN_COLAB:
    if existing_repo_root is not None:
        REPO_ROOT = existing_repo_root
        print(f"Refreshing existing repo at: {REPO_ROOT}")
        run(["git", "-C", str(REPO_ROOT), "fetch", "origin", REPO_BRANCH])
        run(["git", "-C", str(REPO_ROOT), "checkout", REPO_BRANCH])
        run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", REPO_BRANCH])
    else:
        REPO_ROOT = Path("/content") / REPO_NAME
        if not REPO_ROOT.exists():
            print(f"Cloning {REPO_URL} (branch: {REPO_BRANCH})...")
            run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])
        elif not _repo_remote_matches(REPO_ROOT, expected_remote):
            print(f"Remote mismatch detected. Replacing clone...")
            shutil.rmtree(REPO_ROOT)
            run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])
        else:
            print(f"Refreshing existing clone...")
            run(["git", "-C", str(REPO_ROOT), "fetch", "origin", REPO_BRANCH])
            run(["git", "-C", str(REPO_ROOT), "checkout", REPO_BRANCH])
            run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", REPO_BRANCH])

    NOTEBOOK_DIR = REPO_ROOT / "notebooks"
    os.chdir(NOTEBOOK_DIR)
    print(f"Working directory set to: {NOTEBOOK_DIR}")
else:
    REPO_ROOT = NOTEBOOK_DIR.parent

PKG_DIR = REPO_ROOT / "vllm_hook_plugins"
REQ_FILE = REPO_ROOT / "requirement.txt"
FILTERED_REQ_FILE = Path("/tmp/vllm_hook_colab_requirements.txt")
COLAB_RESTART_MARKER = Path("/tmp/vllm_hook_colab_binary_deps_restarted")

print(f"\n--- Environment Summary ---")
print(f"Running in Colab: {IN_COLAB}")
print(f"Repo Root: {REPO_ROOT}")
print(f"Plugin Dir: {PKG_DIR}")

if IN_COLAB:
    assert_cuda_runtime()

# --------------------------------------------------------------------------
# 2. Plugin Directory Validation
# --------------------------------------------------------------------------

if not PKG_DIR.exists():
    raise FileNotFoundError(
        f"Plugin directory not found at {PKG_DIR}. "
        "Please ensure the repository was cloned correctly."
    )

if shutil.which("git") is None and IN_COLAB:
    raise RuntimeError("git is required but unavailable in this runtime.")

# --------------------------------------------------------------------------
# 3. Dependency Management
# --------------------------------------------------------------------------

if REQ_FILE.exists():
    keep = []
    # These packages are managed by Colab's pre-installed environment.
    # Installing them again can cause conflicts.
    blocked = {"vllm", "torch", "torchvision", "torchaudio", "numpy", "scipy", "protobuf"}

    for line in REQ_FILE.read_text().splitlines():
        stripped = line.strip()
        if not stripped or stripped.startswith("#"):
            keep.append(line)
            continue

        package = package_from_req_line(stripped)
        if package in blocked:
            print(f"⏭ Skipping managed dependency: {line}")
            continue
        keep.append(line)

    FILTERED_REQ_FILE.write_text("\n".join(keep) + "\n")
    print(f"Installing filtered requirements from {REQ_FILE.name}...")
    run([sys.executable, "-m", "pip", "install", "-r", str(FILTERED_REQ_FILE)])
else:
    print("⚠ Warning: No requirement.txt found; skipping custom dependency installation.")

# Ensure protobuf is at the correct version
print("Ensuring protobuf version compatibility...")
run([sys.executable, "-m", "pip", "install", "--force-reinstall", "protobuf>=5.29.6,<6.30"])

# --------------------------------------------------------------------------
# 4. vLLM & PyTorch Installation
# --------------------------------------------------------------------------

if COLAB_INSTALL_VLLM:
    print(f"Installing user-specified vLLM: {COLAB_INSTALL_VLLM}")
    run([sys.executable, "-m", "pip", "install", COLAB_INSTALL_VLLM])
else:
    print(f"\nInstalling vLLM and PyTorch for {VLLM_TORCH_BACKEND}...")

    # 1. Uninstall existing conflicting versions
    run([sys.executable, "-m", "pip", "uninstall", "-y", "vllm", "torch", "torchvision", "torchaudio"])

    # 2. Manually remove leftover binary artifacts that pip might miss
    for site_dir in site.getsitepackages():
        site_path = Path(site_dir)
        leftovers = [
            site_path / "vllm", *site_path.glob("vllm-*.dist-info"),
            site_path / "torch", *site_path.glob("torch-*.dist-info"),
            site_path / "torchvision", *site_path.glob("torchvision-*.dist-info"),
            site_path / "torchaudio", *site_path.glob("torchaudio-*.dist-info"),
        ]
        for leftover in leftovers:
            if leftover.exists():
                print(f"  Cleaning up leftover: {leftover.name}")
                if leftover.is_dir():
                    shutil.rmtree(leftover)
                else:
                    leftover.unlink()

    # 3. Force clear GPU memory
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
    except:
        pass

    # 4. Install using 'uv' (faster than pip) with specific CUDA backend
    print("  Downloading and installing packages with 'uv'...")
    run([sys.executable, "-m", "pip", "install", "-U", "uv"])
    run([
        "uv", "pip", "install",
        "--system",
        VLLM_SPEC,
        "torch", "torchvision", "torchaudio",
        f"--torch-backend={VLLM_TORCH_BACKEND}",
    ])

    # 5. Verify Scipy/Numpy
    scipy_check = run_capture([
        sys.executable, "-c", "import numpy, scipy; print('numpy', numpy.__version__); print('scipy', scipy.__version__)"
    ])
    if scipy_check.returncode:
        print("  Detected numpy/scipy issues; upgrading...")
        run([sys.executable, "-m", "pip", "install", "--upgrade", "--force-reinstall", "numpy", "scipy"])

# Verify Installation
print("\nVerifying installation...")
run([
    sys.executable,
    "-c",
    "import torch, vllm; print('✓ torch:', torch.__version__); print('✓ vllm:', getattr(vllm, '__version__', 'unknown'))"
])

# --------------------------------------------------------------------------
# 5. Plugin Loading
# --------------------------------------------------------------------------

# Strategy: Add path to sys.path to allow immediate import,
# then ensure metadata is installed for subprocesses later.
plugin_src_dir = str(PKG_DIR.resolve())
if plugin_src_dir not in sys.path:
    sys.path.insert(0, plugin_src_dir)
importlib.invalidate_caches()

try:
    spec = importlib.util.spec_from_file_location("vllm_hook_plugins", PKG_DIR / "__init__.py")
    if spec:
        importlib.util.module_from_spec(spec)
        print("✓ Plugin module loaded successfully from source path.")
except Exception as e:
    print(f"⚠ Warning: Initial import check failed ({e}). This is expected if compilation is needed later.")

# Ensure the package is registered in the environment (metadata)
# This is necessary for tools that rely on `importlib.metadata`.
print("Registering plugin package in environment...")
run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(PKG_DIR)])

print("Installing W&B tracking client...")
run([sys.executable, "-m", "pip", "install", "wandb"])

print(f"Plugin Source: {plugin_src_dir}")
print(f"Python Exec  : {sys.executable}")

# --------------------------------------------------------------------------
# 6. Runtime Restart (Critical for Colab)
# --------------------------------------------------------------------------
# We restart the runtime to ensure the newly installed binary libraries
# are loaded into a fresh Python interpreter. This prevents "dirty state" issues.
if IN_COLAB and not COLAB_RESTART_MARKER.exists():
    COLAB_RESTART_MARKER.write_text("1\n")
    print("\n" + "="*50)
    print("RESTARTING COLAB RUNTIME...")
    print("This ensures the new vLLM/Torch binaries are fully loaded.")
    print("Do not interrupt this process.")
    print("="*50)

    time.sleep(1) # Allow output buffer to flush
    sys.exit(0) # Terminate this process to trigger Colab restart


Refreshing existing clone...
> Running: git -C /content/vLLM-Hook fetch origin codex/MLR-20-eprime-wandb-integration
From https://github.com/tburleyinfo/vLLM-Hook
 * branch            codex/MLR-20-eprime-wandb-integration -> FETCH_HEAD
> Running: git -C /content/vLLM-Hook checkout codex/MLR-20-eprime-wandb-integration
Already on 'codex/MLR-20-eprime-wandb-integration'
Your branch is up to date with 'origin/codex/MLR-20-eprime-wandb-integration'.
> Running: git -C /content/vLLM-Hook pull --ff-only origin codex/MLR-20-eprime-wandb-integration
From https://github.com/tburleyinfo/vLLM-Hook
 * branch            codex/MLR-20-eprime-wandb-integration -> FETCH_HEAD
Already up to date.
Working directory set to: /content/vLLM-Hook/notebooks

--- Environment Summary ---
Running in Colab: True
Repo Root: /content/vLLM-Hook
Plugin Dir: /content/vLLM-Hook/vllm_hook_plugins
⏭ Skipping managed dependency: vllm>=0.5
⏭ Skipping managed dependency: torch>=2.0
⏭ Skipping managed dependency: numpy>=1.24
In

### Scoring Dependencies

The E-Prime checker uses spaCy for state-of-being verb detection. Run this cell before scoring.


In [2]:
# Optional scoring dependency setup for Colab.
# Run after the main setup cell. If you already have spaCy and en_core_web_sm, this is a no-op.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("spacy") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "spacy"])

try:
    import spacy
    spacy.load("en_core_web_sm", disable=["ner", "parser"])
except OSError:
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])


### Imports & Environment


In [3]:
import gc
import io
import json
import os
import multiprocessing as mp
import re
import subprocess
import sys
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
import torch
from vllm import SamplingParams

if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

from vllm_hook_plugins import HookLLM, generate_with_spotlight, register_plugins
from research.experiment_tracking import (
    ExperimentCondition,
    ExperimentResult,
    RunMetrics,
    TurnResult,
    WandbTracker,
    evaluate_mlr20_condition,
    log_mlr20_results_sequentially,
    run_mlr20_batch,
    collect_runtime_provenance,
    format_gib,
    has_required_vram,
    required_vram_bytes,
    turn_result_from_eprime_row,
)

IN_COLAB = "google.colab" in sys.modules
os.environ["VLLM_USE_V1"] = "1"

if IN_COLAB:
    mp.set_start_method("fork", force=True)
    os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "fork"
    os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
    os.environ.setdefault("HF_HOME", "/content/.cache/huggingface")
    os.environ.setdefault("HUGGINGFACE_HUB_CACHE", "/content/.cache/huggingface/hub")
    os.makedirs(os.environ["HUGGINGFACE_HUB_CACHE"], exist_ok=True)

    def _patch_fileno(stream, fallback_stream, fallback_fd):
        try:
            stream.fileno()
        except io.UnsupportedOperation:
            def _fileno():
                try:
                    return fallback_stream.fileno()
                except Exception:
                    return fallback_fd
            stream.fileno = _fileno

    _patch_fileno(sys.stdout, sys.__stdout__, 1)
    _patch_fileno(sys.stderr, sys.__stderr__, 2)
else:
    mp.set_start_method("spawn", force=True)
    os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

register_plugins()
print("Environment configured")


Environment configured


### C0/C1 Implementation Mode

The mode switch lives before model construction so the Spotlight worker sees the selected behavior during hook installation and request execution.


In [4]:
SPOTLIGHT_IMPLEMENTATION_MODE = os.environ.get("SPOTLIGHT_IMPLEMENTATION_MODE", "C0_GLOBAL_DIAGNOSTIC")
VALID_SPOTLIGHT_IMPLEMENTATION_MODES = {"C0_GLOBAL_DIAGNOSTIC", "C1_PER_QUERY_CONTROL"}
if SPOTLIGHT_IMPLEMENTATION_MODE not in VALID_SPOTLIGHT_IMPLEMENTATION_MODES:
    raise ValueError(f"Unsupported SPOTLIGHT_IMPLEMENTATION_MODE={SPOTLIGHT_IMPLEMENTATION_MODE!r}")

print(f"Initial Spotlight implementation mode: {SPOTLIGHT_IMPLEMENTATION_MODE}")
print("Batch execution sets this mode per condition: C0 -> C0_GLOBAL_DIAGNOSTIC, C1 -> C1_PER_QUERY_CONTROL")


Initial Spotlight implementation mode: C0_GLOBAL_DIAGNOSTIC
Batch execution sets this mode per condition: C0 -> C0_GLOBAL_DIAGNOSTIC, C1 -> C1_PER_QUERY_CONTROL


### Patch Spotlight Bias At The Aggregation Boundary

This patch preserves the canonical global path for C0 while logging the per-query span-attention proportions that canonical Spotlight currently aggregates away. C1 uses the same per-query proportions as its branch point.


In [5]:
import torch.nn.functional as F
from vllm_hook_plugins.workers import spotlight_worker as spotlight_worker_module
from vllm_hook_plugins.utils import spotlight as spotlight_utils_package
from vllm_hook_plugins.utils.spotlight import utils as spotlight_utils_module

spotlight_query_diagnostics = []

def compute_spotlight_bias_c0_c1(logits, span_ranges, target_proportion):
    """C0/C1 Spotlight bias helper with a shared per-query intermediate.

    C0 logs psi_current(i) and then follows canonical global Spotlight.
    C1 starts from the same psi_current(i) vector and applies query-indexed control.
    """
    attn_weights = F.softmax(logits, dim=-1)
    modified_weights = attn_weights.clone()

    for batch_idx, ranges in enumerate(span_ranges):
        if not ranges:
            continue

        union_mask = torch.zeros(
            modified_weights.size(-1),
            device=modified_weights.device,
            dtype=modified_weights.dtype,
        )
        for start, end in ranges:
            union_mask[start:end] = 1.0
        union_mask = union_mask.view(1, 1, -1)

        span_mass_by_query = (modified_weights[batch_idx] * union_mask).sum(dim=(0, 2))
        total_mass_by_query = modified_weights[batch_idx].sum(dim=(0, 2)).clamp_min(1e-12)
        psi_current_by_query = span_mass_by_query / total_mass_by_query

        spotlight_query_diagnostics.append({
            "mode": SPOTLIGHT_IMPLEMENTATION_MODE,
            "target_proportion": float(target_proportion),
            "psi_current_by_query": psi_current_by_query.detach().cpu().float().tolist(),
            "psi_current_global": float((modified_weights[batch_idx] * union_mask).sum().detach().cpu() / modified_weights[batch_idx].sum().detach().cpu()),
        })

        if SPOTLIGHT_IMPLEMENTATION_MODE == "C0_GLOBAL_DIAGNOSTIC":
            current_proportion = (modified_weights[batch_idx] * union_mask).sum() / modified_weights[batch_idx].sum()
            if current_proportion < target_proportion:
                bias_value = torch.log(torch.tensor(target_proportion / current_proportion, device=modified_weights.device, dtype=torch.float32))
                attn_logits = logits[batch_idx].float() + union_mask * bias_value
                modified_weights[batch_idx] = F.softmax(attn_logits, dim=-1, dtype=torch.float32).to(modified_weights.dtype)
            continue

        if SPOTLIGHT_IMPLEMENTATION_MODE == "C1_PER_QUERY_CONTROL":
            needs_steer = psi_current_by_query < target_proportion
            if torch.any(needs_steer):
                safe_psi = psi_current_by_query.clamp_min(1e-12)
                bias_by_query = torch.where(
                    needs_steer,
                    torch.log(torch.full_like(safe_psi, float(target_proportion)) / safe_psi),
                    torch.zeros_like(safe_psi),
                ).view(1, -1, 1)
                attn_logits = logits[batch_idx].float() + union_mask * bias_by_query
                modified_weights[batch_idx] = F.softmax(attn_logits, dim=-1, dtype=torch.float32).to(modified_weights.dtype)

    return modified_weights

spotlight_utils_module.compute_spotlight_bias = compute_spotlight_bias_c0_c1
spotlight_utils_package.compute_spotlight_bias = compute_spotlight_bias_c0_c1
spotlight_worker_module.compute_spotlight_bias = compute_spotlight_bias_c0_c1
print("Patched Spotlight bias helper for C0/C1 boundary diagnostics")


Patched Spotlight bias helper for C0/C1 boundary diagnostics


### GPU Memory Preflight


In [6]:
def show_nvidia_smi():
    if not IN_COLAB:
        return
    try:
        completed = subprocess.run(
            ["nvidia-smi"],
            check=False,
            capture_output=True,
            text=True,
        )
    except FileNotFoundError:
        print("nvidia-smi unavailable in this runtime.")
        return
    if completed.stdout.strip():
        print(completed.stdout)
    if completed.stderr.strip():
        print(completed.stderr)


def cuda_memory_preflight(gpu_memory_utilization, *, show_smi=True):
    gc.collect()
    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA is unavailable. In Colab, select a GPU runtime, restart the runtime, "
            "and run the notebook from the beginning."
        )

    torch.cuda.empty_cache()
    if hasattr(torch.cuda, "ipc_collect"):
        try:
            torch.cuda.ipc_collect()
        except Exception as exc:
            print(f"torch.cuda.ipc_collect() skipped: {exc}")

    free_bytes, total_bytes = torch.cuda.mem_get_info()
    required_bytes = required_vram_bytes(total_bytes, gpu_memory_utilization)
    print(
        "CUDA memory after cleanup: "
        f"free={format_gib(free_bytes)} total={format_gib(total_bytes)} "
        f"required={format_gib(required_bytes)} "
        f"for gpu_memory_utilization={gpu_memory_utilization:.2f}"
    )

    if show_smi:
        show_nvidia_smi()

    if not has_required_vram(
        free_bytes=free_bytes,
        total_bytes=total_bytes,
        gpu_memory_utilization=gpu_memory_utilization,
    ):
        raise RuntimeError(
            "Not enough free CUDA memory for the configured vLLM reservation. "
            f"Free={format_gib(free_bytes)}, required={format_gib(required_bytes)}, "
            f"total={format_gib(total_bytes)}, "
            f"gpu_memory_utilization={gpu_memory_utilization:.2f}. "
            "Restart the Colab runtime and choose Runtime > Run all. "
            "Do not continue from a partially executed runtime."
        )

    return {
        "free_bytes": free_bytes,
        "total_bytes": total_bytes,
        "required_bytes": required_bytes,
    }


### Initialize `HookLLM`


In [7]:
cache_dir = "/content/.cache/vllm-hook" if IN_COLAB else os.path.expanduser("~/.cache/vllm-hook")
model = "Qwen/Qwen2-1.5B-Instruct"
MAX_MODEL_LEN = 8192
GPU_MEMORY_UTILIZATION = float(os.environ.get("VLLM_HOOK_GPU_MEMORY_UTILIZATION", "0.30" if IN_COLAB else "0.65"))

gpu_memory_preflight = cuda_memory_preflight(GPU_MEMORY_UTILIZATION)

llm = HookLLM(
    model=model,
    worker_name="probe_spotlight",
    download_dir=cache_dir,
    trust_remote_code=True,
    dtype=torch.float16,
    enable_hook=True,
    gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
    max_model_len=MAX_MODEL_LEN,
    max_num_seqs=1,
    enforce_eager=True,
    enable_prefix_caching=False,
    enable_chunked_prefill=False,
    tensor_parallel_size=1,
)

print(f"Model loaded: {model}")
print("Spotlight worker enabled")
print(f"Max model length: {MAX_MODEL_LEN}")
print(f"GPU memory utilization: {GPU_MEMORY_UTILIZATION}")


CUDA memory after cleanup: free=54.82 GiB total=79.25 GiB required=23.78 GiB for gpu_memory_utilization=0.30
Thu Sep 24 01:30:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   33C    P0             61W /  400W |   25017MiB /  81920MiB |      0%      Default |
|                              

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


WARNING 09-24 01:30:15 [arg_utils.py:1352] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
INFO 09-24 01:30:16 [model.py:533] Resolved architecture: Qwen2ForCausalLM
WARNING 09-24 01:30:16 [model.py:1920] Casting torch.bfloat16 to torch.float16.
INFO 09-24 01:30:16 [model.py:1582] Using max model len 8192
WARNING 09-24 01:30:16 [arg_utils.py:2080] This model does not officially support disabling chunked prefill. Disabling this manually may cause the engine to crash or produce incorrect outputs.
INFO 09-24 01:30:16 [vllm.py:775] Asynchronous scheduling is enabled.
WARNING 09-24 01:30:16 [vllm.py:809] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 09-24 01:30:16 [vllm.py:820] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor 

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


INFO 09-24 01:30:24 [default_loader.py:384] Loading weights took 1.52 seconds
INFO 09-24 01:30:26 [gpu_model_runner.py:4566] Model loading took 2.89 GiB memory and 3.351303 seconds
INFO 09-24 01:30:28 [gpu_worker.py:456] Available KV cache memory: 20.26 GiB
INFO 09-24 01:30:28 [kv_cache_utils.py:1316] GPU KV cache size: 758,848 tokens
INFO 09-24 01:30:28 [kv_cache_utils.py:1321] Maximum concurrency for 8,192 tokens per request: 92.63x
INFO 09-24 01:30:28 [core.py:281] init engine (profile, create kv cache, warmup model) took 2.51 seconds
INFO 09-24 01:30:31 [llm.py:391] Supported tasks: ('generate',)
Model loaded: Qwen/Qwen2-1.5B-Instruct
Spotlight worker enabled
Max model length: 8192
GPU memory utilization: 0.3


### E-Prime Long-Conversation Configuration


In [8]:
E_PRIME_CONSTRAINT = """Persistent E-Prime constraint:
Do not use state-of-being verbs: am, is, are, was, were, be, being, been.
Do not use contractions containing state-of-being verbs, including I'm, you're, we're, they're, he's, she's, it's, that's, and there's.
""".strip()

SYSTEM_MESSAGE = (
    "You are a concise assistant continuing a long Talk2AI-style conversation. "
    "Preserve the E-Prime constraint throughout the whole conversation while still answering helpfully."
)

SEED_HISTORY = [
    {"role": "user", "content": "I keep hearing that climate change discussions are exaggerated compared with other problems."},
    {"role": "assistant", "content": "Let's compare long-run measurements with the way headlines describe them."},
    {"role": "user", "content": "The headlines make me distrust the whole topic."},
    {"role": "assistant", "content": "Repeated alarm can make even strong evidence feel performative."},
]

USER_TURNS = [
    "What evidence should I look at if I want to avoid headline-driven conclusions?",
    "How do local weather experiences confuse the broader trend?",
    "What tradeoffs matter for households when policies raise energy costs?",
    "Where does adaptation make sense, and where does it fall short?",
    "How should poorer countries think about growth and emissions limits?",
    "What practical local policy question should I ask a city council candidate?",
    "How can I separate serious policy criticism from misinformation?",
    "What should climate communicators stop doing if they want skeptical people to listen?",
    "What should good-faith skeptics concede before debating policy?",
    "End with a cautious, practical position that still respects the evidence.",
]

ALPHA = 0.2
SAMPLING_PARAMS = SamplingParams(temperature=0.0, max_tokens=160)
MAX_USER_TURNS = min(10, len(USER_TURNS))
HISTORY_WINDOW_MESSAGES = 16


### Prompt And Run Helpers


In [9]:
def render_e_prime_prompt(history, user_message):
    recent_history = history[-HISTORY_WINDOW_MESSAGES:]
    omitted_messages = max(0, len(history) - len(recent_history))
    transcript = "\n".join(
        f"{item['role'].upper()}: {item['content']}" for item in recent_history
    )
    if transcript:
        transcript += "\n"

    earlier_context = ""
    if omitted_messages:
        earlier_context = (
            f"Earlier conversation context: {omitted_messages} older messages are omitted "
            "from this prompt to keep the Colab run within memory limits. Continue the same conversation.\n\n"
        )

    return f"""{SYSTEM_MESSAGE}

{E_PRIME_CONSTRAINT}

{earlier_context}Recent conversation:
{transcript}USER: {user_message}
ASSISTANT:""".strip()



STATE_OF_BEING_CORE_WORDS = {"am", "is", "are", "was", "were", "be", "being", "been"}
STATE_OF_BEING_SPACY_MODEL = "en_core_web_sm"
CONTRACTION_PATTERN = re.compile(
    r"\b(?:i'm|you're|we're|they're|he's|she's|it's|that's|there's|what's|who's|where's|when's|why's|how's)\b",
    re.IGNORECASE,
)


def load_state_of_being_nlp(model_name=STATE_OF_BEING_SPACY_MODEL):
    try:
        import spacy
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "spaCy is required for the E-Prime checker. Run the scoring dependency setup cell first."
        ) from exc

    try:
        return spacy.load(model_name, disable=["ner", "parser"])
    except OSError as exc:
        raise OSError(
            f"spaCy model {model_name!r} is not installed. Run the scoring dependency setup cell first."
        ) from exc


state_of_being_nlp = load_state_of_being_nlp()


def is_state_of_being_token(token):
    lemma = token.lemma_.lower()
    text = token.text.lower()
    if lemma == "be" and token.pos_ in {"AUX", "VERB"}:
        return True
    return text in STATE_OF_BEING_CORE_WORDS and token.pos_ in {"AUX", "VERB"}


def check_e_prime_violations(text, nlp=state_of_being_nlp):
    text = text or ""
    doc = nlp(text)
    verb_matches = []
    for token in doc:
        if is_state_of_being_token(token):
            verb_matches.append(
                {
                    "text": token.text,
                    "lemma": token.lemma_,
                    "pos": token.pos_,
                    "tag": token.tag_,
                    "start": token.idx,
                    "end": token.idx + len(token.text),
                }
            )
    contraction_matches = [
        {"text": match.group(0), "start": match.start(), "end": match.end()}
        for match in CONTRACTION_PATTERN.finditer(text)
    ]
    violation_count = len(verb_matches) + len(contraction_matches)
    return {
        "state_of_being_count": len(verb_matches),
        "contraction_count": len(contraction_matches),
        "e_prime_violation_count": violation_count,
        "e_prime_retained": violation_count == 0,
        "e_prime_score": 1.0 if violation_count == 0 else 0.0,
        "state_of_being_matches": verb_matches,
        "contraction_matches": contraction_matches,
    }


def rows_from_experiment_results(experiment_results):
    rows = []
    for experiment_result in experiment_results:
        rows.extend(experiment_result.full_result["rows"])
    return rows


### Run One C-Series Condition Or C0/C1 Batch


In [10]:
C_SERIES_COMPARISON_GROUP = "MLR-20-C-aggregation-resolution"
C_SERIES_NOTEBOOK = "notebooks/demo_spotlight_e_prime_C0_C1_colab.ipynb"
C_SERIES_ALPHA = 0.10

C_SERIES_COMMON = dict(
    comparison_group=C_SERIES_COMPARISON_GROUP,
    spotlight=True,
    alpha=C_SERIES_ALPHA,
    model=model,
    constraint_formulation="Full E-Prime",
    constraint_complexity="Negative enumeration",
    intervention_timing="Prefill intervention",
    history="Self-propagating history",
    turns=MAX_USER_TURNS,
    replicate=1,
    temperature=SAMPLING_PARAMS.temperature,
    max_tokens=SAMPLING_PARAMS.max_tokens,
    history_window_messages=HISTORY_WINDOW_MESSAGES,
    seed=None,
    notebook=C_SERIES_NOTEBOOK,
    tags=("MLR-20", "C-series", "aggregation-resolution", "notion-design-matrix"),
)

C_SERIES_CONDITIONS = [
    ExperimentCondition(
        condition_id="C0",
        implementation="global_diagnostic",
        extra={
            "notion_experiment": "C0 Global aggregation + per-query diagnostic alpha 0.10",
            "notion_comparison_group": "C — Aggregation resolution",
            "notion_implementation": "Global aggregation with per-query diagnostic logging",
            "aggregation": "Global aggregation",
            "diagnostic": "Record per-query psi_current(i) immediately before canonical global aggregation",
            "spotlight_implementation_mode": "C0_GLOBAL_DIAGNOSTIC",
        },
        **C_SERIES_COMMON,
    ),
    ExperimentCondition(
        condition_id="C1",
        implementation="per_query_control",
        extra={
            "notion_experiment": "C1 Per-query control alpha 0.10",
            "notion_comparison_group": "C — Aggregation resolution",
            "notion_implementation": "Per-query control",
            "aggregation": "Query-indexed control",
            "diagnostic": "Branch from the same per-query psi_current(i) intermediate used by C0",
            "spotlight_implementation_mode": "C1_PER_QUERY_CONTROL",
        },
        **C_SERIES_COMMON,
    ),
]

conditions_by_id = {condition.condition_id: condition for condition in C_SERIES_CONDITIONS}
print("Loaded C-series Spotlight/E-Prime conditions:")
for condition in C_SERIES_CONDITIONS:
    print(
        f"  {condition.condition_id}: alpha={condition.alpha} "
        f"implementation={condition.implementation} "
        f"mode={condition.extra['spotlight_implementation_mode']}"
    )

RUN_SINGLE_CONDITION_DEBUG = os.environ.get("MLR20_RUN_SINGLE_CONDITION_DEBUG", "0") == "1"
SELECTED_CONDITION_ID = os.environ.get("MLR20_SELECTED_CONDITION_ID", "C0")
selected_condition_result = None
if RUN_SINGLE_CONDITION_DEBUG:
    selected_condition = conditions_by_id[SELECTED_CONDITION_ID]
    SPOTLIGHT_IMPLEMENTATION_MODE = selected_condition.extra["spotlight_implementation_mode"]
    selected_condition_result = evaluate_mlr20_condition(
        condition=selected_condition,
        llm=llm,
        sampling_params=SAMPLING_PARAMS,
        seed_history=SEED_HISTORY,
        user_turns=USER_TURNS,
        render_prompt=render_e_prime_prompt,
        score_fn=check_e_prime_violations,
        generate_with_spotlight_fn=generate_with_spotlight,
        spotlight_span=E_PRIME_CONSTRAINT,
        provenance=collect_runtime_provenance(REPO_ROOT),
    )
    print(f"Single-condition debug run complete: {SELECTED_CONDITION_ID}")

batch_started = time.perf_counter()
batch_provenance = collect_runtime_provenance(REPO_ROOT)
c_series_batch_results = []
for condition in C_SERIES_CONDITIONS:
    SPOTLIGHT_IMPLEMENTATION_MODE = condition.extra["spotlight_implementation_mode"]
    print(f"Running {condition.condition_id} with {SPOTLIGHT_IMPLEMENTATION_MODE}")
    c_series_batch_results.append(
        evaluate_mlr20_condition(
            condition=condition,
            llm=llm,
            sampling_params=SAMPLING_PARAMS,
            seed_history=SEED_HISTORY,
            user_turns=USER_TURNS,
            render_prompt=render_e_prime_prompt,
            score_fn=check_e_prime_violations,
            generate_with_spotlight_fn=generate_with_spotlight,
            spotlight_span=E_PRIME_CONSTRAINT,
            provenance=batch_provenance,
        )
    )
batch_elapsed_s = time.perf_counter() - batch_started
results = pd.DataFrame(rows_from_experiment_results(c_series_batch_results))
print(f"C0/C1 batch complete in {batch_elapsed_s:.2f}s across {len(c_series_batch_results)} conditions")
results


Loaded C-series Spotlight/E-Prime conditions:
  C0: alpha=0.1 implementation=global_diagnostic mode=C0_GLOBAL_DIAGNOSTIC
  C1: alpha=0.1 implementation=per_query_control mode=C1_PER_QUERY_CONTROL
Running C0 with C0_GLOBAL_DIAGNOSTIC


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

...

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

C0/C1 batch complete in 28.22s across 2 conditions


,condition,turn,prompt,user_message,model_response,compliant,violation_count,state_of_being_count,contraction_count,score_text_source,prompt_chars,history_messages_in_prompt,latency_s,state_of_being_matches,contraction_matches,reply,e_prime_score,e_prime_retained,e_prime_violation_count
0,C0,1,You are a concise assistant continuing a long ...,What evidence should I look at if I want to av...,Look for studies that have been peer-reviewed ...,False,1,1,0,assistant_response,845,4,0.802383,"[{'text': 'been', 'lemma': 'be', 'pos': 'AUX',...",[],Look for studies that have been peer-reviewed ...,0.0,False,1
1,C0,2,You are a concise assistant continuing a long ...,How do local weather experiences confuse the b...,Local weather experiences can be influenced by...,False,1,1,0,assistant_response,1125,6,1.006636,"[{'text': 'be', 'lemma': 'be', 'pos': 'AUX', '...",[],Local weather experiences can be influenced by...,0.0,False,1
2,C0,3,You are a concise assistant continuing a long ...,What tradeoffs matter for households when poli...,Energy costs can have significant impacts on h...,False,2,1,1,assistant_response,1484,8,1.247389,"[{'text': ''s', 'lemma': 'be', 'pos': 'AUX', '...","[{'text': 'It's', 'start': 253, 'end': 257}]",Energy costs can have significant impacts on h...,0.0,False,2
3,C0,4,You are a concise assistant continuing a long ...,"Where does adaptation make sense, and where do...",Adaptation can make sense in areas where the c...,False,7,6,1,assistant_response,1925,10,1.656496,"[{'text': 'is', 'lemma': 'be', 'pos': 'AUX', '...","[{'text': 'It's', 'start': 276, 'end': 280}]",Adaptation can make sense in areas where the c...,0.0,False,7
4,C0,5,You are a concise assistant continuing a long ...,How should poorer countries think about growth...,Poorer countries may face unique challenges in...,False,3,2,1,assistant_response,2441,12,1.416889,"[{'text': ''s', 'lemma': 'be', 'pos': 'AUX', '...","[{'text': 'It's', 'start': 281, 'end': 285}]",Poorer countries may face unique challenges in...,0.0,False,3
5,C0,6,You are a concise assistant continuing a long ...,What practical local policy question should I ...,What steps will the candidate take to reduce g...,False,1,1,0,assistant_response,2964,14,0.698058,"[{'text': 'are', 'lemma': 'be', 'pos': 'AUX', ...",[],What steps will the candidate take to reduce g...,0.0,False,1
6,C0,7,You are a concise assistant continuing a long ...,How can I separate serious policy criticism fr...,It's important to distinguish between serious ...,False,12,10,2,assistant_response,3222,16,1.819577,"[{'text': ''s', 'lemma': 'be', 'pos': 'AUX', '...","[{'text': 'It's', 'start': 0, 'end': 4}, {'tex...",It's important to distinguish between serious ...,0.0,False,12
7,C0,8,You are a concise assistant continuing a long ...,What should climate communicators stop doing i...,It's important for climate communicators to be...,False,8,7,1,assistant_response,3758,16,1.915016,"[{'text': ''s', 'lemma': 'be', 'pos': 'AUX', '...","[{'text': 'It's', 'start': 0, 'end': 4}]",It's important for climate communicators to be...,0.0,False,8
8,C0,9,You are a concise assistant continuing a long ...,What should good-faith skeptics concede before...,It's important for good-faith skeptics to ackn...,False,3,2,1,assistant_response,4212,16,1.692283,"[{'text': ''s', 'lemma': 'be', 'pos': 'AUX', '...","[{'text': 'It's', 'start': 0, 'end': 4}]",It's important for good-faith skeptics to ackn...,0.0,False,3
9,C0,10,You are a concise assistant continuing a long ...,"End with a cautious, practical position that s...",It's important for policymakers and communicat...,False,7,4,3,assistant_response,4459,16,1.846705,"[{'text': ''s', 'lemma': 'be', 'pos': 'AUX', '...","[{'text': 'It's', 'start': 0, 'end': 4}, {'tex...",It's important for policymakers and communicat...,0.0,False,7


### Validate E-Prime Scoring Scope


In [11]:
if "reply" not in results.columns:
    raise ValueError("Expected generated assistant replies in the results table.")
if not set(["e_prime_score", "e_prime_violation_count", "state_of_being_matches", "contraction_matches"]).issubset(results.columns):
    raise ValueError("Expected assistant-only E-Prime scores from the condition evaluator.")
print("E-Prime compliance was scored only on newly generated assistant replies.")
results[["condition", "turn", "e_prime_score", "e_prime_violation_count", "reply"]].head()


E-Prime compliance was scored only on newly generated assistant replies.


,condition,turn,e_prime_score,e_prime_violation_count,reply
0,C0,1,0.0,1,Look for studies that have been peer-reviewed ...
1,C0,2,0.0,1,Local weather experiences can be influenced by...
2,C0,3,0.0,2,Energy costs can have significant impacts on h...
3,C0,4,0.0,7,Adaptation can make sense in areas where the c...
4,C0,5,0.0,3,Poorer countries may face unique challenges in...


### Compare Constraint Retention


In [12]:
summary = results.groupby("condition").agg(
    mean_e_prime_retention=("e_prime_score", "mean"),
    violation_rate=("e_prime_retained", lambda values: 1 - values.mean()),
    mean_violations=("e_prime_violation_count", "mean"),
    mean_state_of_being_count=("state_of_being_count", "mean"),
    mean_contraction_count=("contraction_count", "mean"),
    mean_latency_s=("latency_s", "mean"),
    max_prompt_chars=("prompt_chars", "max"),
)
summary


,mean_e_prime_retention,violation_rate,mean_violations,mean_state_of_being_count,mean_contraction_count,mean_latency_s,max_prompt_chars
condition,,,,,,,
C0,0.0,1.0,4.5,3.5,1.0,1.410143,4459
C1,0.0,1.0,4.5,3.5,1.0,1.390876,4459


### Inspect Failures


In [13]:
failures = results[results["e_prime_score"] < 1.0][
    [
        "condition",
        "turn",
        "user_message",
        "e_prime_violation_count",
        "state_of_being_matches",
        "contraction_matches",
        "reply",
    ]
]
failures


,condition,turn,user_message,e_prime_violation_count,state_of_being_matches,contraction_matches,reply
0,C0,1,What evidence should I look at if I want to av...,1,"[{'text': 'been', 'lemma': 'be', 'pos': 'AUX',...",[],Look for studies that have been peer-reviewed ...
1,C0,2,How do local weather experiences confuse the b...,1,"[{'text': 'be', 'lemma': 'be', 'pos': 'AUX', '...",[],Local weather experiences can be influenced by...
2,C0,3,What tradeoffs matter for households when poli...,2,"[{'text': ''s', 'lemma': 'be', 'pos': 'AUX', '...","[{'text': 'It's', 'start': 253, 'end': 257}]",Energy costs can have significant impacts on h...
3,C0,4,"Where does adaptation make sense, and where do...",7,"[{'text': 'is', 'lemma': 'be', 'pos': 'AUX', '...","[{'text': 'It's', 'start': 276, 'end': 280}]",Adaptation can make sense in areas where the c...
4,C0,5,How should poorer countries think about growth...,3,"[{'text': ''s', 'lemma': 'be', 'pos': 'AUX', '...","[{'text': 'It's', 'start': 281, 'end': 285}]",Poorer countries may face unique challenges in...
5,C0,6,What practical local policy question should I ...,1,"[{'text': 'are', 'lemma': 'be', 'pos': 'AUX', ...",[],What steps will the candidate take to reduce g...
6,C0,7,How can I separate serious policy criticism fr...,12,"[{'text': ''s', 'lemma': 'be', 'pos': 'AUX', '...","[{'text': 'It's', 'start': 0, 'end': 4}, {'tex...",It's important to distinguish between serious ...
7,C0,8,What should climate communicators stop doing i...,8,"[{'text': ''s', 'lemma': 'be', 'pos': 'AUX', '...","[{'text': 'It's', 'start': 0, 'end': 4}]",It's important for climate communicators to be...
8,C0,9,What should good-faith skeptics concede before...,3,"[{'text': ''s', 'lemma': 'be', 'pos': 'AUX', '...","[{'text': 'It's', 'start': 0, 'end': 4}]",It's important for good-faith skeptics to ackn...
9,C0,10,"End with a cautious, practical position that s...",7,"[{'text': ''s', 'lemma': 'be', 'pos': 'AUX', '...","[{'text': 'It's', 'start': 0, 'end': 4}, {'tex...",It's important for policymakers and communicat...


### Save Results And Limitations


In [14]:
out_dir = Path("/content/spotlight_e_prime_results") if IN_COLAB else Path("notebooks/results")
out_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
csv_path = out_dir / f"spotlight_e_prime_C0_C1_constraint_retention_{stamp}.csv"
json_path = out_dir / f"spotlight_e_prime_C0_C1_constraint_retention_{stamp}.json"

results.to_csv(csv_path, index=False)
json_path.write_text(
    json.dumps(
        {
            "model": model,
            "conditions": [condition.to_config() for condition in C_SERIES_CONDITIONS],
            "constraint": E_PRIME_CONSTRAINT,
            "metric": "e_prime_score == 1.0 when no state-of-being verbs or listed contractions appear",
            "max_user_turns": MAX_USER_TURNS,
            "history_window_messages": HISTORY_WINDOW_MESSAGES,
            "max_model_len": MAX_MODEL_LEN,
            "gpu_memory_utilization": GPU_MEMORY_UTILIZATION,
            "summary": summary.reset_index().to_dict(orient="records"),
            "limitations": [
                "Notebook smoke validation checks structure and platform-specific imports only unless this notebook is run in Colab.",
                "The contraction checker uses an explicit list and may not catch every informal contraction.",
                "The spaCy POS-based checker can miss malformed output or mis-tag unusual phrasing.",
            ],
            "rows": results.to_dict(orient="records"),
            "wandb_run_invariant": "one C-series Experimental Conditions row = one ExperimentCondition = one W&B run",
        },
        indent=2,
    ),
    encoding="utf-8",
)

print(f"Saved CSV: {csv_path}")
print(f"Saved JSON: {json_path}")
print("Validation note: this committed notebook has been smoke-validated locally; run it in a Colab GPU runtime for execution results.")


Saved CSV: /content/spotlight_e_prime_results/spotlight_e_prime_C0_C1_constraint_retention_20260924-013101.csv
Saved JSON: /content/spotlight_e_prime_results/spotlight_e_prime_C0_C1_constraint_retention_20260924-013101.json
Validation note: this committed notebook has been smoke-validated locally; run it in a Colab GPU runtime for execution results.


### Track C0/C1 As Separate W&B Runs


In [16]:
ENABLE_WANDB = os.environ.get("WANDB_MODE", "").lower() not in {"disabled", "off"}
WANDB_PROJECT = os.environ.get("WANDB_PROJECT", "vllm-hook-eprime")

# Optional one-time paste for notebook runs. Leave blank to use the environment
# or Colab secret named WANDB_API_KEY. Do not save/share the notebook with a key filled in.
WANDB_API_KEY = ""

if ENABLE_WANDB:
    if WANDB_API_KEY.strip():
        os.environ["WANDB_API_KEY"] = WANDB_API_KEY.strip()
    elif IN_COLAB:
        try:
            from google.colab import userdata
            wandb_api_key = userdata.get("WANDB_API_KEY")
            if wandb_api_key:
                os.environ["WANDB_API_KEY"] = wandb_api_key
        except Exception:
            pass

for experiment_result in c_series_batch_results:
    experiment_result.provenance.update(
        {
            "results_csv": str(csv_path),
            "results_json": str(json_path),
            "batch_elapsed_s": batch_elapsed_s,
            "batch_condition_count": len(c_series_batch_results),
            "gpu_memory_utilization": GPU_MEMORY_UTILIZATION,
            "max_model_len": MAX_MODEL_LEN,
        }
    )
    experiment_result.full_result.update(
        {
            "model": model,
            "constraint": E_PRIME_CONSTRAINT,
            "metric": "e_prime_score == 1.0 when no state-of-being verbs or listed contractions appear",
            "max_user_turns": MAX_USER_TURNS,
            "history_window_messages": HISTORY_WINDOW_MESSAGES,
            "max_model_len": MAX_MODEL_LEN,
            "gpu_memory_utilization": GPU_MEMORY_UTILIZATION,
            "condition_config": experiment_result.condition.to_config(),
            "limitations": [
                "Notebook smoke validation checks structure and platform-specific imports only unless this notebook is run in Colab.",
                "The contraction checker uses an explicit list and may not catch every informal contraction.",
                "The spaCy POS-based checker can miss malformed output or mis-tag unusual phrasing.",
            ],
        }
    )

wandb_payloads = log_mlr20_results_sequentially(
    c_series_batch_results,
    project=WANDB_PROJECT,
    enabled=ENABLE_WANDB,
)

print(f"Prepared {len(wandb_payloads)} C-series W&B payloads")
if ENABLE_WANDB:
    print(f"Logged one W&B run per condition in project={WANDB_PROJECT}")
else:
    print("W&B logging disabled; payloads were built locally for validation.")
wandb_payloads[0]["config"]


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: tm8ctgzqj8 (tm8ctgzqj8-georgia-institute-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


compliance_rate,▁
first_violation_turn,▁
mean_state_of_being_count,▁
mean_violations,▁
total_contractions,▁
total_violations,▁
compliance_rate,0
first_violation_turn,1
mean_state_of_being_count,3.5
mean_violations,4.5
total_contractions,10


compliance_rate,▁
first_violation_turn,▁
mean_state_of_being_count,▁
mean_violations,▁
total_contractions,▁
total_violations,▁
compliance_rate,0
first_violation_turn,1
mean_state_of_being_count,3.5
mean_violations,4.5
total_contractions,10


Prepared 2 C-series W&B payloads
Logged one W&B run per condition in project=vllm-hook-eprime


{'condition_id': 'C0',
 'comparison_group': 'MLR-20-C-aggregation-resolution',
 'spotlight': True,
 'alpha': 0.1,
 'implementation': 'global_diagnostic',
 'model': 'Qwen/Qwen2-1.5B-Instruct',
 'constraint_formulation': 'Full E-Prime',
 'constraint_complexity': 'Negative enumeration',
 'intervention_timing': 'Prefill intervention',
 'history': 'Self-propagating history',
 'turns': 10,
 'replicate': 1,
 'temperature': 0.0,
 'max_tokens': 160,
 'history_window_messages': 16,
 'seed': None,
 'notebook': 'notebooks/demo_spotlight_e_prime_C0_C1_colab.ipynb',
 'notion_experiment': 'C0 Global aggregation + per-query diagnostic alpha 0.10',
 'notion_comparison_group': 'C — Aggregation resolution',
 'notion_implementation': 'Global aggregation with per-query diagnostic logging',
 'aggregation': 'Global aggregation',
 'diagnostic': 'Record per-query psi_current(i) immediately before canonical global aggregation',
 'spotlight_implementation_mode': 'C0_GLOBAL_DIAGNOSTIC',
 'git_sha': 'a704b0dfb6348